# Teste de Extração: PAM (Produção Agrícola Municipal)

Extrai dados de lavouras temporárias (1612) e permanentes (1613) via API SIDRA.

## URLs da API SIDRA (março/2026)

| Tabela | Descrição | URL Padrão |
|--------|-----------|------------|
| **1612** | Lavoura Temporária | `https://apisidra.ibge.gov.br/values/t/1612/n6/all/v/all/p/{ANO}` |
| **1613** | Lavoura Permanente | `https://apisidra.ibge.gov.br/values/t/1613/n6/all/v/all/p/{ANO}` |

## Status dos Dados

- **2020-2024:** Dados completos disponíveis
- **2025:** Não disponível no nível municipal (apenas estimativas mensais LSPA)
- **Colunas:** D1C (ano), D1N, D2C (município), D2N, D3C (produto), D3N, ..., V (valor)

## Exemplo de URL Direta

```text
https://apisidra.ibge.gov.br/values/t/1612/n6/all/v/all/p/2024
```

## Troubleshooting

| Problema | Solução |
|----------|----------|
| Erro 503 | Retry com backoff exponencial |
| sidrapy 404 | Usar requests direto (biblioteca abandonada) |
| Rate limit | Máximo 2 workers + sleep |
| Dados vazios | Ano sem dados disponíveis |

In [1]:
import requests
import pandas as pd
import uuid
import time
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List

# ====================== CONFIG ======================
DATA_DIR = Path("data")
BRONZE_DIR = DATA_DIR / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

class EscritorParquet:
    def __init__(self, diretorio_base: Path):
        self.diretorio_base = diretorio_base

    def escrever_chunk(self, df: pd.DataFrame, nome_dataset: str, coluna_particao: str = None):
        if df is None or df.empty:
            return
        destino = self.diretorio_base / nome_dataset
        if coluna_particao and coluna_particao in df.columns:
            valor = df[coluna_particao].iloc[0]
            destino = destino / f"{coluna_particao}={valor}"
            df = df.drop(columns=[coluna_particao])
        destino.mkdir(parents=True, exist_ok=True)
        df.to_parquet(destino / f"chunk_{uuid.uuid4().hex[:8]}.parquet", index=False)

# ====================== EXTRATOR ROBUSTO ======================
class ExtratorPAM:
    def __init__(self, anos: List[int]):
        self.anos = anos
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Python-PAM-Downloader'
        })

    def _baixar_ano(self, ano: int) -> pd.DataFrame:
        """Baixa dados de um ano específico com retry automático."""
        dfs = []
        for tab, tipo in [("1612", "temporaria"), ("1613", "permanente")]:
            url = f"https://apisidra.ibge.gov.br/values/t/{tab}/n6/all/v/all/p/{ano}"
            for tentativa in range(5):
                try:
                    r = self.session.get(url, timeout=60)
                    r.raise_for_status()
                    data = r.json()
                    
                    if not data or not isinstance(data, list) or len(data) <= 1:
                        break  # ano sem dados
                    
                    df = pd.DataFrame(data)
                    df["tipo_lavoura"] = tipo
                    dfs.append(df)
                    break
                    
                except Exception as e:
                    print(f"Tentativa {tentativa+1}/5 - {tab} {ano}: {e}")
                    time.sleep(2 ** tentativa)  # backoff exponencial
            else:
                print(f"❌ Falha total ao baixar {tab} {ano}")
        
        return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    def extrair(self):
        # Máximo 2 workers + backoff = respeita rate limit do IBGE
        with ThreadPoolExecutor(max_workers=2) as ex:
            futures = {ex.submit(self._baixar_ano, a): a for a in self.anos}
            for f in tqdm(as_completed(futures), total=len(self.anos), desc="Baixando PAM"):
                yield f.result()

# ====================== EXECUÇÃO ======================
if __name__ == "__main__":
    anos = list(range(2020, 2025))          # ← já inclui 2024 (dados mais recentes)
    extrator = ExtratorPAM(anos)
    escritor = EscritorParquet(BRONZE_DIR)

    for chunk in extrator.extrair():
        if not chunk.empty:
            part = "D1C" if "D1C" in chunk.columns else None
            escritor.escrever_chunk(chunk, "pam", coluna_particao=part)
    
    print("✅ Concluído! Todos os anos salvos em data/bronze/pam/")

Baixando PAM: 100%|██████████| 5/5 [02:42<00:00, 32.55s/it]

✅ Concluído! Todos os anos salvos em data/bronze/pam/
